# Vehicle Classifier — test the final model (Colab)

This notebook is only a thin front end. All the work is done by `run_test.py` in the repository,
the same script that runs from the command line. Full instructions: `docs/MENTOR_TEST_GUIDE.md`.

**You need three things on Google Drive (or uploaded to Colab):**
1. the project code (this repository, as a folder or a `.zip`), or access to the GitHub repository;
2. the model file `swin_t_ft_aug_best.pt` (about 105 MB) and the Neysan detector `neysan_detector.json`;
3. your test images: a folder (or a `.zip`) with one subfolder per class
   (`ambulance/ autobus/ kamyun/ kamyunet/ minibus/ savari/ taxi/ vanet/`).
   Images without class folders also work; then only predictions are written.

Images the Neysan detector recognises as a Neysan are reported as `unknown` (see `docs/NEYSAN_REJECTION.md`).

Edit the paths in the next cell, then run all cells (*Runtime → Run all*). A GPU is optional.

In [ ]:
# ---- edit these paths ----
REPO_SOURCE = "https://github.com/alidarviish/vehicle-classifier.git"   # git URL, or a Drive folder / .zip of the repository
CHECKPOINT  = "/content/drive/MyDrive/vehicle_classifier/swin_t_ft_aug_best.pt"
NEYSAN_DETECTOR = "/content/drive/MyDrive/vehicle_classifier/neysan_detector.json"
TEST_INPUT  = "/content/drive/MyDrive/vehicle_classifier/TEST"            # a folder or a .zip file
OUTPUT      = "/content/drive/MyDrive/vehicle_classifier/results"         # must not exist yet (or be empty)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# get the code
import os, shutil, subprocess, zipfile
REPO = "/content/vehicle-classifier"
if not os.path.isdir(REPO):
    if REPO_SOURCE.startswith(("http://", "https://", "git@")):
        subprocess.run(["git", "clone", "--depth", "1", REPO_SOURCE, REPO], check=True)
    elif REPO_SOURCE.lower().endswith(".zip"):
        with zipfile.ZipFile(REPO_SOURCE) as z:
            z.extractall("/content/repo_unzip")
        inner = [os.path.dirname(p) for p in zipfile.ZipFile(REPO_SOURCE).namelist() if p.endswith("run_test.py")]
        assert inner, "run_test.py not found inside the zip"
        shutil.move(os.path.join("/content/repo_unzip", min(inner, key=len)), REPO)
    else:
        shutil.copytree(REPO_SOURCE, REPO)
assert os.path.isfile(os.path.join(REPO, "run_test.py")), "run_test.py not found - check REPO_SOURCE"
print("code ready:", REPO)

In [ ]:
# get the test images (a .zip is extracted to the local Colab disk, which is faster than reading from Drive)
TEST_DIR = TEST_INPUT
if TEST_INPUT.lower().endswith(".zip"):
    TEST_DIR = "/content/test_images"
    if not os.path.isdir(TEST_DIR):
        with zipfile.ZipFile(TEST_INPUT) as z:
            bad = z.testzip()
            assert bad is None, f"the zip file is damaged (first bad entry: {bad}); upload it again"
            z.extractall(TEST_DIR)
assert os.path.isdir(TEST_DIR), f"test folder not found: {TEST_DIR}"
assert os.path.isfile(CHECKPOINT), f"checkpoint not found: {CHECKPOINT}"
assert os.path.isfile(NEYSAN_DETECTOR), f"Neysan detector not found: {NEYSAN_DETECTOR}"
print("test images:", TEST_DIR)

In [ ]:
# run the test (the same command as on a local machine)
!cd "{REPO}" && python run_test.py --input "{TEST_DIR}" --checkpoint "{CHECKPOINT}" --neysan-detector "{NEYSAN_DETECTOR}" --output "{OUTPUT}"

In [ ]:
# show the results
from IPython.display import Image, display
print(open(os.path.join(OUTPUT, "summary.txt"), encoding="utf-8").read())
png = os.path.join(OUTPUT, "confusion_matrix.png")
if os.path.isfile(png):
    display(Image(png))
print("result files:", sorted(os.listdir(OUTPUT)))